## MongoDB Code Inconsistency Database Builder

This notebook will build the code inconsistency database. It uses the code generation database, so do build the code generation database first before building this database.

The new database will contain processed data in a new schema used specifically for code inconsistency testing. 

In [1]:
import os
import sys
import copy
from typing import List, Any
import inspect
from tqdm import tqdm
import pandas as pd
import ast

In [2]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [3]:
from database import MongoDBHelper
from code_inconsistency.utility.codesense_helper import CodeInconsistencyCodeSenseHelper

In [4]:
db = MongoDBHelper()
if db.check_database_connectivity():
    print("MongoDB connected")

MongoDB connected


In [5]:
base_qns_db = db.client["Base_Questions_DB"]
cruxeval_database = pd.read_csv(
    "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/datasets/open_ended_format/CodeSense/loop_iteration_dataset_python.csv",
    encoding="utf-8",
    header=0,
    quoting=1,           
    )

In [6]:
tf_question_database = base_qns_db["CodeSense_Input_Output"]

In [9]:
c = set()
tot = 0
repurposed_qn = {}
rej_tot = 0
for i in tqdm(range(
    # 1,2
    # 33,34
    len(cruxeval_database)
    )):
    task_id = f"CodeSenseTF{i}"
    sample_qn = cruxeval_database.iloc[i]
    full_sol : str = sample_qn['scratchpad_format']
    prompt = sample_qn['question']
    expected_output = sample_qn['answer']
    original_id = sample_qn['idx']

    sol_lines = full_sol.splitlines()
    func_call_line = sol_lines.pop()
    full_sol = "\n".join(sol_lines)
    
    func_name, task_input = CodeInconsistencyCodeSenseHelper.extract_task_input(func_call_line)
    line_num = CodeInconsistencyCodeSenseHelper.find_for_loop_target_line_num(prompt)
    modified_code = CodeInconsistencyCodeSenseHelper.modify_full_sol(full_sol, line_num)

    namespace = {}
    # print(func_name)
    # print(modified_code)
    # print(func_call_line)
    try: 
        exec(modified_code, namespace)
        counter_instance = namespace['CounterClass']()
    except Exception as e:
        print(original_id, "failed at modification")
    try:
        result = getattr(counter_instance, func_name)(*task_input)
    except Exception as e:
        print(original_id, f"failed to RUN {e}")

        continue
    if counter_instance.counter != eval(expected_output):
        print(original_id, "failed")



    

  0%|          | 0/107 [00:00<?, ?it/s]

100%|██████████| 107/107 [00:00<00:00, 1051.41it/s]

1 failed to RUN "'__name__' not in globals"
2 failed to RUN "'__name__' not in globals"
3 failed to RUN "'__name__' not in globals"
5 failed to RUN name 'GenericSubtitleParser' is not defined
6 failed to RUN name 'GenericSubtitleParser' is not defined
7 failed to RUN name 'git_refnames' is not defined
8 failed to RUN name 'ast' is not defined
10 failed to RUN No module named 'schema'
11 failed to RUN name 'os' is not defined
12 failed at modification
12 failed to RUN 'CounterClass' object has no attribute 'to_dict'
13 failed at modification
13 failed to RUN 'CounterClass' object has no attribute 'parse_all'
14 failed to RUN name 'COCO_CATEGORIES' is not defined
15 failed to RUN name 'wrap' is not defined
17 failed to RUN name 'numeric_start' is not defined
18 failed to RUN name 'numeric_start' is not defined
19 failed to RUN name 'less' is not defined
20 failed to RUN name 'util' is not defined
21 failed to RUN name 'cos_open' is not defined
22 failed at modification
22 failed to RUN '

In [8]:
c = set()
tot = 0
repurposed_qn = {}
rej_tot = 0
for i in tqdm(range(
    len(cruxeval_database)
    )):
    task_id = f"CodeSenseTF{i}"
    sample_qn = cruxeval_database.iloc[i]
    full_sol = sample_qn['scratchpad_format']
    prompt = sample_qn['question']
    expected_output = sample_qn['answer']
    original_id = sample_qn['idx']

    namespace = {}
    test_input = CodeInconsistencyCodeSenseHelper.extract_task_input(full_sol)
    print(test_input)
    x
    func_name  = CodeInconsistencyCodeSenseHelper.extract_func_name(full_sol)

    exec(full_sol, namespace)

    try:
        test_input = eval(test_input, namespace) if not isinstance(test_input, float) else None
        expected_output = eval(expected_output)
    except Exception as e:
        print(test_input)
        print(task_id, f"failed due to following error: {e}")
        continue

    sig = inspect.signature(namespace[func_name])
    input_copy = copy.deepcopy(test_input) if isinstance(test_input, (list, dict, tuple, set)) else test_input
    try:
        if len(sig.parameters) == 0:
            assert namespace[func_name]() == expected_output
        elif len(sig.parameters) > 1:
            assert namespace[func_name](*input_copy) == expected_output
        else:
            assert namespace[func_name](input_copy) == expected_output

    except:
        print(f"Did not pass test case. Double check task_id {task_id}, test_case {test_input}")
    
    input_metadata = type(test_input).__name__

    func_in_input = None

    if input_metadata not in ('str', 'NoneType'):
        input_args_tree = ast.parse(sample_qn['input'])
        func_in_input = any(isinstance(node, (ast.Call, ast.Lambda)) for node in ast.walk(input_args_tree))

    if isinstance(test_input, dict):
        test_input = str(test_input)
    elif isinstance(test_input, (tuple, list)):
        test_input = str(test_input) if any(i for i in test_input if isinstance(i, dict)) else test_input
    
    ### Storing / updating entry in the database
    db_entry = {
        "_id" : task_id,
        "full_sol" : full_sol,
        "input" : {
            "args": test_input if not func_in_input else sample_qn['input'],
            "metadata": input_metadata,
            },
        "output": {
            "args" : str(expected_output),
            "metadata": type(expected_output).__name__,
            },
        "original_id": original_id
    }

    try:
        tf_question_database.update_one(
            filter={"_id": task_id},
            update={"$set": db_entry},
            upsert=True
        )
    except Exception as e:
        print(f"Could not enter test case {original_id} into TF database due to the following error: {e}")


    ## Secondary check where the question is pulled from the database and tested against the check function
    ## This step is necessary as MongoDB does not store these details in standard Python data formats and a secondary step is needed for sanity check. 
    ## For example, tuples are stored as "arrays" in MongoDB, which are converted to Lists in Python.
    
    qn = tf_question_database.find_one({"_id" : task_id})
    if qn is None:
        continue
    full_sol = qn['full_sol']                           # full canonical solution for the task

    test_inputs = qn['input']                           # unpacking input args and metadata from qn
    input_args = test_inputs['args']                    # test input args
    input_metadata = test_inputs['metadata']            # test input metadata

    test_outputs = qn['output']                         # unpacking outputs args and metadata from qn
    output_args = test_outputs['args']                  # test output args
    output_metadata = test_outputs['metadata']          # test output metadata

    try:
        input_args = eval(input_args) if isinstance(input_args, str) and input_metadata != str.__name__  else input_args
    except Exception as e:
        print(original_id)
        print(e)
        continue
    output_args = ast.literal_eval(output_args) if output_metadata != str.__name__ else output_args

    check_stored_soln_validity = CodeInconsistencyCruxEvalHelper.check_input_output(
        full_sol=full_sol,
        test_input=input_args,
        input_metadata=input_metadata,
        expected_output= output_args,
        func_name= func_name,
    )

    if check_stored_soln_validity is not True:
        tf_question_database.find_one_and_delete({"_id" : task_id})
        print(f'{original_id} from {task_id} failed the secondary checks and was not added to the database.')
        tot-=1


  0%|          | 0/107 [00:00<?, ?it/s]


('find', [[{'Variable': 'jenkins_admin_password', 'Type': 'password'}, {'Variable': 'ca_rootca_password', 'Type': 'password'}], 'Variable', 'something_not_there'])


NameError: name 'x' is not defined

sample_452: modified such that the "," behind the initial test input is removed and the single quotation marks replaced with double

In [ ]:
print(f"{tf_question_database.count_documents({})} total test cases in the database")
print(f'{tot} valid test cases')
print(f'{rej_tot} test cases were rejected')

800 total test cases in the database
0 valid test cases
0 test cases were rejected


# Sanity Check for Code Inconsistency Test Cases in MongoDB [Deprecated]

In [ ]:
%%script false --no-raise-error
import os
import sys
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)
from code_inconsistency.utility.humaneval_helper import CodeInconsistencyHumanEvalHelper
from database import MongoDBHelper
from tqdm import tqdm 

db = MongoDBHelper()
if db.check_database_connectivity():
    print("MongoDB connected")
base_qns_db = db.client["Base_Questions_DB"]
question_database = base_qns_db["HumanEval_Input_Output"]

for num in tqdm(range(
    question_database.count_documents({})
    )):
    try:
        task_id = f"HumanEvalTF{num}"
        qn = question_database.find_one({"_id": task_id})

        full_sol = qn['full_sol']                           # full canonical solution for the task
        qn_desc = qn['qn_desc']                             # task description. This should be the extracted doc string from the original task
        examples = qn['examples']                           # examples for other prompt techniques like one shot, few shot

        test_inputs = qn['input']                           # unpacking input args and metadata from qn
        input_args = test_inputs['args']                    # test input args
        input_metadata = test_inputs['metadata']            # test input metadata

        test_outputs = qn['output']                         # unpacking outputs args and metadata from qn
        output_args = test_outputs['args']                  # test output args
        output_metadata = test_outputs['metadata']          # test output metadata

        random_test_case = list(examples.keys())[0]
        func_name = CodeInconsistencyHumanEvalHelper.extract_func_name_from_example(random_test_case)      

        if output_metadata == type(None).__name__:
            output_metadata = "type(None)"
        if not eval(output_metadata) == str:
            output_args = eval(output_args)

        check_soln_validity = CodeInconsistencyHumanEvalHelper.check_input_output(
            full_sol= full_sol,
            test_input= input_args,
            expected_output= output_args,
            func_name=func_name,
            input_metadata = input_metadata
        )
        if not check_soln_validity:
            print(task_id)
    except Exception as e:
        print(f"{task_id}, {e}")
